In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
from matplotlib import pyplot as plt

In [2]:
data = pd.read_csv("all_chem_df.csv")

data["tags"] = data["tags"].fillna("").str.split()
data = data.drop(["image_name", "Col3"], axis=1)
data = data[sorted(data.columns)]

data = data.explode("tags")
data

,smiles,tags
0,CC(=O)NC1C(O)OC(CO)C(O)C1O,dermatologic
1,CCC[C@@]1(CCc2ccccc2)CC(O)=C([C@H](CC)c2cccc(N...,antiinfective
2,CCCCC(C)C(=O)OC1C(C)C(CC)OC2(CC3CC(C/C=C(\C)CC...,antiinfective
3,COc1cc2c(c(OC)c1OC)-c1c(cc3c(c1OC)OCO3)C[C@H](...,antineoplastic
4,CC(=O)N[C@@H](CS)C(=O)[O-],antiinfective
...,...,...
8331,CC(=O)Oc1ccccc1C(=O)O.OCCN(CCO)c1nc(N2CCCCC2)c...,hematologic
8332,C=CO.C=O,hematologic
8333,CC1(C)SC2[C@H](NC(=O)[C@H](N)c3ccccc3)C(=O)N2[...,antiinfective
8334,COCCCOc1ccnc(C[S@@](=O)c2nc3ccccc3[nH]2)c1C,gastrointestinal


In [3]:
import json

import requests
import pandas as pd
import urllib3

urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning) #To silence the warning

baseUrl = 'https://admetlab3.scbdd.com'


def transform(data):
    resultList = []
    for mol in data['data']:
        if not mol['data']:
            # Invalid SMILES
            tmp = {'smiles': mol['smiles']}
        else:
            tmp = dict({'smiles': mol['smiles']})
            for _, admet in mol['data'].items():
                for endpoint in admet:
                    # endpoint is a dict
                    tmp[endpoint['name']] = endpoint['value']
        resultList.append(tmp)
    return pd.DataFrame(resultList).fillna('Invalid SMILES')


def divide_list(lst, n):
    for i in range(0, len(lst), n):
        yield lst[i:i + n]


if __name__ == '__main__':
    api = '/api/admet'
    url = baseUrl + api
    param = {
        'SMILES': []
    }
    n = 1000
    smiles_list = ['CN1C2CCC1CC(OC(=O)c1cccn1C)C2', 'O=C(O)Nc1scnc1C(=O)Nc1nccs1',
                   'COc1ccc(C=C(F)C(=O)c2cc(OC)c(OC)c(OC)c2)cc1'] * 2500

    for _, sublist in enumerate(divide_list(smiles_list, n)):
        param['SMILES'] = sublist

        response = requests.post(url, json=param, verify=False)

        if response.status_code == 200:  # If access is successful
            data = response.json()['data']
            # transform to csv file
            result = transform(data)
            result.to_csv('result' + str(_) + '.csv', index=False)
            
        else:
            print("error")

error
error
error
error
error
error
error
error


In [4]:
test_output_df = pd.read_csv("ADMET_test_output.csv")
test_output_df

,raw_smiles,smiles,MW,Vol,Dense,nHA,nHD,TPSA,nRot,nRing,...,Acute_Aquatic_Toxicity,FAF-Drugs4 Rule,Genotoxic_Carcinogenicity_Mutagenicity,Aggregators,Fluc,Blue_fluorescence,Green_fluorescence,Reactive,Other_assay_interference,Promiscuous
0,CC(=O)NC1C(O)OC(CO)C(O)C1O,CC(=O)NC1C(O)OC(CO)C(O)C1O,221.09,199.469545,1.108390,7,5,119.25,3,1,...,['-'],['-'],['-'],0.001,0.000,0.018,0.063,0.020,0.433,0.000
1,CCC[C@@]1(CCc2ccccc2)CC(O)=C([C@H](CC)c2cccc(N...,CCC[C@@]1(CCc2ccccc2)CC(O)=C([C@H](CC)c2cccc(N...,602.21,584.161474,1.030896,7,2,105.59,12,4,...,"[(13, 15, 39, 40, 41, 3), (13, 15, 39, 40, 41)...","[(24, 23, 22, 21, 20, 19, 38), (13, 15, 39)]","[(13, 15, 39, 40, 41), (23, 24)]",0.999,0.008,0.126,0.901,0.024,0.590,0.001
2,CCCCC(C)C(=O)OC1C(C)C(CC)OC2(CC3CC(C/C=C(\C)CC...,CCCCC(C)C(=O)OC1C(C)C(CC)OC2(CC3CC(C/C=C(\C)CC...,686.40,712.401104,0.963502,10,3,140.98,7,5,...,"[(12,), (36, 35, 34), (40, 41, 42, 43, 18)]","[(6, 8, 9, 10, 12, 15, 16, 17, 18, 43, 41)]","[(30, 31, 32, 33)]",0.446,0.001,0.006,0.316,0.220,0.163,0.001
3,COc1cc2c(c(OC)c1OC)-c1c(cc3c(c1OC)OCO3)C[C@H](...,COc1cc2c(c(OC)c1OC)-c1c(cc3c(c1OC)OCO3)C[C@H](...,416.18,417.851118,0.996001,7,1,75.61,4,4,...,"[(28, 26, 24), (4, 29)]","[(21, 20, 16, 17, 12, 13, 14, 15, 22)]",['-'],0.166,0.322,0.427,0.345,0.142,0.448,0.536
4,CC(=O)N[C@@H](CS)C(=O)[O-],CC(=O)N[C@@H](CS)C(=O)O,163.03,145.639933,1.119405,4,2,66.40,4,0,...,['-'],"[(5, 6)]",['-'],0.181,0.000,0.008,0.021,0.532,0.865,0.082
5,CC(=O)OCC(=O)C1CCC2C3CCC4CC(O)CCC4(C)C3C(=O)CC12C,CC(=O)OCC(=O)C1CCC2C3CCC4CC(O)CCC4(C)C3C(=O)CC12C,390.24,408.180041,0.956049,5,1,80.67,4,4,...,"[(17, 16, 15), (4, 5, 6, 7)]","[(4, 5, 7, 6)]",['-'],0.090,0.000,0.010,0.006,0.228,0.002,0.018
6,CC(=O)Nc1nnc(S(N)(=O)=O)s1,CC(=O)Nc1nnc(S(N)(=O)=O)s1,221.99,168.650336,1.316274,7,3,115.04,3,1,...,['-'],['-'],"[(4, 3, 1, 2, 0), (4,), (4, 3)]",0.052,0.039,0.066,0.111,0.298,0.418,0.991
7,O=C([O-])CCC/C=C\C[C@H]1[C@H]2CC[C@H](C2)[C@@H...,O=C(O)CCC/C=C\C[C@H]1[C@H]2CC[C@H](C2)[C@@H]1N...,377.17,380.291185,0.991793,5,2,83.47,9,3,...,['-'],['-'],['-'],0.011,0.000,0.019,0.008,0.039,0.754,0.062
8,OCCN1CCN(CC/C=C2\c3ccccc3Sc3ccc(Cl)cc32)CC1,OCCN1CCN(CC/C=C2\c3ccccc3Sc3ccc(Cl)cc32)CC1,400.14,400.890926,0.998127,3,1,26.71,5,4,...,"[(21, 22)]",['-'],['-'],0.902,0.006,0.191,0.305,0.038,0.529,0.630
9,C[C@@H]1CCC/C=C/[C@H]2C[C@@H](O)C[C@@H]2[C@@H]...,C[C@@H]1CCC/C=C/[C@H]2C[C@@H](O)C[C@@H]2[C@@H]...,280.17,295.430836,0.948344,4,2,66.76,0,2,...,"[(15, 16, 17, 18, 19, 1), (15, 16, 17, 18, 19)...","[(15, 16, 17)]","[(15, 16, 17, 18, 19)]",0.530,0.090,0.079,0.000,0.554,0.058,0.480
